# 300장 실험 준비 (채점표 + 검색 DB 만들기)

위에서부터 **순서대로** 실행하세요. 바꿔야 하는 곳은 **2번 셀의 zip 경로**뿐이에요.

| 순서 | 하는 일 |
|---|---|
| 1 | 구글 드라이브 연결 + 코드 최신으로 받기 |
| 2 | zip 경로 적기 (★ 여기만 수정) |
| 3 | zip 풀기 |
| 4 | 채점표 `manifest.csv` 만들기 |
| 5 | 검색 DB `search_db.csv` 만들기 |
| 6 | 결과 미리보기 |
| 7 | 결과를 드라이브에 저장 (런타임이 끊겨도 남게) |

In [1]:
# 1) 구글 드라이브 연결 + 코드 최신으로 받기
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/dl_deep_imgps'):
    %cd /content/dl_deep_imgps
    !git pull
else:
    %cd /content
    !git clone https://github.com/yewon51019/dl_deep_imgps
    %cd /content/dl_deep_imgps
!ls exp300

Mounted at /content/drive
/content
Cloning into 'dl_deep_imgps'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 125 (delta 39), reused 108 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 6.06 MiB | 23.23 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/dl_deep_imgps
make_manifest.py  make_search_db.py  run_exp300_prepare.ipynb


In [4]:
!unzip -q "/content/drive/MyDrive/Colab Notebooks/pill_images_test/약제각인인식_테스트셋 (1).zip" -d /content/data1
!unzip -q "/content/drive/MyDrive/Colab Notebooks/pill_images_test/약제각인인식_테스트셋.zip" -d /content/data2
!ls /content/data1 /content/data2

/content/data1:
약제각인인식_테스트셋

/content/data2:
약제각인인식_테스트셋


In [2]:
# 2) ★ zip 파일 경로 적기 (드라이브 왼쪽 파일창에서 zip 우클릭 -> '경로 복사' 후 붙여넣기)
ZIP_FILES = [
    '/content/drive/MyDrive/약제각인인식_테스트셋.zip',
    '/content/drive/MyDrive/약제각인인식_테스트셋(1).zip',
]

# (선택) AI Hub 전체 라벨 폴더가 드라이브에 있으면 적기. 없으면 빈 칸 [] 그대로 두기.
#   -> 검색 DB 후보를 27종이 아니라 실제처럼 많게 만들 때 사용
EXTRA_LABEL_DIRS = []

# 개발용 비율 (0.2 = 20%, 0 이면 나누지 않음)
DEV_RATIO = 0.2

for z in ZIP_FILES:
    print('있음 ✓' if os.path.exists(z) else '없음 ✗ (경로 확인!)', z)

없음 ✗ (경로 확인!) /content/drive/MyDrive/약제각인인식_테스트셋.zip
없음 ✗ (경로 확인!) /content/drive/MyDrive/약제각인인식_테스트셋(1).zip


In [5]:
# 3) zip 풀기 (zip 마다 다른 폴더에)
import subprocess
UNZIP_DIRS = []
for i, z in enumerate(ZIP_FILES, 1):
    out = f'/content/data{i}'
    if not os.path.exists(out):
        subprocess.run(['unzip', '-q', '-o', z, '-d', out], check=True)
    UNZIP_DIRS.append(out)

# images/ 와 labels/ 가 같이 들어있는 폴더를 자동으로 찾기
ROOTS = []
for d in UNZIP_DIRS:
    for cur, subdirs, files in os.walk(d):
        if 'images' in subdirs and 'labels' in subdirs:
            ROOTS.append(cur)
print('찾은 폴더:'); [print(' ', r) for r in ROOTS]

찾은 폴더:
  /content/data1/약제각인인식_테스트셋
  /content/data2/약제각인인식_테스트셋


[None, None]

In [6]:
# 4) 채점표 manifest.csv 만들기 (이미지 1장당 1줄, 정답 포함)
subprocess.run(['python', 'exp300/make_manifest.py', '--roots', *ROOTS,
                '--out', '/content/manifest.csv', '--dev-ratio', str(DEV_RATIO)])

CompletedProcess(args=['python', 'exp300/make_manifest.py', '--roots', '/content/data1/약제각인인식_테스트셋', '/content/data2/약제각인인식_테스트셋', '--out', '/content/manifest.csv', '--dev-ratio', '0.2'], returncode=0)

In [7]:
# 5) 검색 DB search_db.csv 만들기 (약 1종당 1줄)
label_dirs = [os.path.join(r, 'labels') for r in ROOTS] + EXTRA_LABEL_DIRS
subprocess.run(['python', 'exp300/make_search_db.py', '--label-roots', *label_dirs,
                '--out', '/content/search_db.csv'])

CompletedProcess(args=['python', 'exp300/make_search_db.py', '--label-roots', '/content/data1/약제각인인식_테스트셋/labels', '/content/data2/약제각인인식_테스트셋/labels', '--out', '/content/search_db.csv'], returncode=0)

In [8]:
# 6) 결과 미리보기
import pandas as pd
m = pd.read_csv('/content/manifest.csv')
db = pd.read_csv('/content/search_db.csv')
print('채점표', m.shape, '/ 검색 DB', db.shape)
display(m[['file_name','drug_N','dl_name','form','drug_dir','camera_la','rot_deg','truth','split']].head(10))
display(db[['drug_N','dl_name','form','shape','color1','print_front','print_back']])

채점표 (126, 21) / 검색 DB (11, 13)


,file_name,drug_N,dl_name,form,drug_dir,camera_la,rot_deg,truth,split
0,K-013392_0_2_0_2_90_000_200.png,K-013392,스피드펜연질캡슐200mg/PTP,연질캡슐제,앞면,90,0,SPEED,test
1,K-013392_0_2_0_2_90_060_200.png,K-013392,스피드펜연질캡슐200mg/PTP,연질캡슐제,앞면,90,60,SPEED,test
2,K-013392_0_2_0_2_90_120_200.png,K-013392,스피드펜연질캡슐200mg/PTP,연질캡슐제,앞면,90,120,SPEED,test
3,K-013392_0_2_0_2_90_180_200.png,K-013392,스피드펜연질캡슐200mg/PTP,연질캡슐제,앞면,90,180,SPEED,dev
4,K-013392_0_2_0_2_90_240_200.png,K-013392,스피드펜연질캡슐200mg/PTP,연질캡슐제,앞면,90,240,SPEED,test
5,K-013392_0_2_0_2_90_300_200.png,K-013392,스피드펜연질캡슐200mg/PTP,연질캡슐제,앞면,90,300,SPEED,test
6,K-017057_0_0_0_2_60_000_200.png,K-017057,환인아캄프로세이트정(아캄프로세이트칼슘),장용정제,앞면,60,0,AC,test
7,K-017057_0_0_0_2_60_060_200.png,K-017057,환인아캄프로세이트정(아캄프로세이트칼슘),장용정제,앞면,60,60,AC,test
8,K-017057_0_0_0_2_60_120_200.png,K-017057,환인아캄프로세이트정(아캄프로세이트칼슘),장용정제,앞면,60,120,AC,test
9,K-017057_0_0_0_2_60_180_200.png,K-017057,환인아캄프로세이트정(아캄프로세이트칼슘),장용정제,앞면,60,180,AC,test


,drug_N,dl_name,form,shape,color1,print_front,print_back
0,K-013392,스피드펜연질캡슐200mg/PTP,연질캡슐제,타원형,"노랑, 투명",SPEED,NaN
1,K-017057,환인아캄프로세이트정(아캄프로세이트칼슘),장용정제,원형,하양,AC,NaN
2,K-019196,래피콜에이캡슐 100mg/PTP,경질캡슐제,장방형,주황,YH RPC-A,NaN
3,K-020395,벤라팩트서방캡슐,서방캡슐제,장방형,분홍,SCD VFS,NaN
4,K-021930,심발타캡슐 30mg,장용캡슐제,장방형,파랑,9543 30mg,NaN
5,K-021931,심발타캡슐 60mg,장용캡슐제,장방형,파랑,9542 60mg,NaN
6,K-023336,아리셉트에비스정 5mg,"확산정, 속붕정, 구강붕해정",원형,하양,ARICEPT,5.0
7,K-023337,아리셉트에비스정 10mg,"확산정, 속붕정, 구강붕해정",원형,노랑,ARICEPT,10.0
8,K-023405,레믹실오디티정 30mg,"확산정, 속붕정, 구강붕해정",원형,하양,TX,NaN
9,K-024650,벤라팩트서방캡슐 37.5mg,서방캡슐제,장방형,회색,SCD 37.5,NaN


In [9]:
# 7) 결과를 드라이브에 저장 (런타임이 끊겨도 남도록)
SAVE_DIR = '/content/drive/MyDrive/exp300_results'
os.makedirs(SAVE_DIR, exist_ok=True)
!cp /content/manifest.csv /content/search_db.csv "{SAVE_DIR}"
!ls -l "{SAVE_DIR}"

total 38
-rw------- 1 root root 35640 Oct 10 06:47 manifest.csv
-rw------- 1 root root  2154 Oct 10 06:47 search_db.csv
